In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatOllama

# 设定系统上下文，构建提示词
template = """请扮演一位资深的技术博主，您将负责为用户生成适合在微博发布的中文文章。
           请把用户输入的内容扩展成140个字左右的文章，并且添加适当的表情符号使内容引人入胜并体现专业性。"""
prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", "{input}")])

# 通过 Ollama 加载Llama 2 13B对话补全模型
model = ChatOllama(model="llama2-chinese:13b",
                     base_url="http://localhost:22515",
                     temperature=0.7,
                     max_tokens=512)

# 通过LCEL构建调用链并执行，得到文本输出
chain = prompt | model | StrOutputParser()
chain.invoke({"input": "给大家推荐一本新书《LangChain实战》，让我们一起学习LangChain吧！"})

'📚👀💻好想和大家分享最近读到的好书——《LangChain实战》，它是关于LangChain技术的专业性较高的书籍。\n\n作为一名深入研究LangChain技术的人，我想分享这本书对我的影响和对LangChain技术的认识。首先，这本书非常易于理解，包含了大量的实例和示例，使得阅读过程变得更加生动化。其次，它还涵盖了LangChain技术在不同应用场景中的应用，从而为读者提供了更广泛的视野和思路方向。\n\n通过阅读这本书，我对LangChain技术有了更深入的认识，并希望大家也能够从中得益。如果你对LangChain技术感兴趣，那么这本书是一个不错的选择。\n\n最后，我想说一下关于作者和出版社的信息：这本书由王尧、杨倩等人联合编写，并由机械哔哩出版有限公司出版发行。希望大家能够收到这本书的阅读体验，并与我一起进入LangChain技术的世界！\n'

In [6]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)
formatted_prompt = prompt_template.format(adjective="funny", content="chickens")

print(formatted_prompt)

Tell me a funny joke about chickens.


'Tell me a funny joke about chickens.'

In [7]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI bot. Your name is {name}."),
    ("human", "Hello, how are you doing?"),
    ("ai", "I'm doing well, thanks!"),
    ("human", "{user_input}")
])

chat_template.format_messages(name="Bob", user_input="what is your name?")

[SystemMessage(content='You are a helpful AI bot. Your name is Bob.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is your name?', additional_kwargs={}, response_metadata={})]

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.example_selectors import LengthBasedExampleSelector

examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input": "windy", "output": "calm"},
]

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 设置期望的示例文本长度
    max_length=25,
)

dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    # 设置示例以外部分的前置文本
    prefix="Give the antonym of every input",
    # 设置示例以外部分的后置文本
    suffix="Input: {adjective}\nOutput:\n\n",
    input_variables=["adjective"],
)

# 当用户输入的内容比较短时，所有示例都会被引用
print(dynamic_prompt.format(adjective="big"))

# 当用户输入的内容比较长时，只有少量示例会被引用
long_astring = "big and huge and gigantic and massive and colossus and towering and lofty and elevated and much much much much bigger than everything else"
print(dynamic_prompt.format(adjective=long_astring))

Give the antonym of every input

Input: happy
Output: sad

Input: tall
Output: short

Input: energetic
Output: lethargic

Input: sunny
Output: gloomy

Input: windy
Output: calm

Input: big
Output:


Give the antonym of every input

Input: big and huge and gigantic and massive and colossus and towering and lofty and elevated and much much much much bigger than everything else
Output:




In [21]:
%pip install pydantic

from typing import List
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_community.llms.ollama import Ollama
from langchain_core.output_parsers import PydanticOutputParser
import re

class Actor(BaseModel):
    name: str = Field(description="The name of an author")
    book_names: List[str] = Field(description="list of names of books they wrote")
    
# 明确要求仅输出一个符合 schema 的 JSON 实例，避免模型返回 schema 本身或额外说明
actor_query = "请随机生成一位知名的80年代中文作家及其代表作品，并仅以一个符合下面 schema 的 JSON 对象输出，不要输出 schema 或任何解释。"

parser = PydanticOutputParser(pydantic_object=Actor)

prompt = PromptTemplate(
    template="请回答下面的问题：\n{query}\n\n{format_instructions}\n如果输出是代码块，请不要包含首尾的```符号。\n请只输出一个完整的 JSON 对象示例，不要其他多余文字。",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

input = prompt.format_prompt(query=actor_query)
print("Prompt to model:\n", input)

model = Ollama(model="llama2-chinese:13b"
                     , base_url="http://localhost:22515"
                     , temperature=0.7)

# 请求模型生成
result = model.generate([input.to_string()])
output_text = result.generations[0][0].text
print("Raw model output:\n", output_text)

# 如果模型附带说明或重复了 schema，尝试提取第一个 JSON 对象部分
m = re.search(r"\{.*\}", output_text, re.S)
json_text = m.group(0) if m else output_text.strip()

# 解析并返回 pydantic 对象（捕获并打印解析错误以便调试）
try:
    actor = parser.parse(json_text)
    print("Parsed actor object:\n", actor)
except Exception as e:
    print("Failed to parse output into Actor:", e)
    print("Extracted JSON text was:\n", json_text)
    raise

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
Prompt to model:
 text='请回答下面的问题：\n请随机生成一位知名的80年代中文作家及其代表作品，并仅以一个符合下面 schema 的 JSON 对象输出，不要输出 schema 或任何解释。\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "The name of an author", "title": "Name", "type": "string"}, "book_names": {"description": "list of names of books they wrote", "items": {"type": "string"}, "title": "Book Names", "type": "array"}}, "required": ["name", "book_names"]}\n```\n如果输出是代码块，请不要包含首尾的```符号。